In [0]:
df = spark.table("cafe_sales_2026.silver.clean_cafe_sales")
display(df)

In [0]:
from pyspark.sql import functions as F

# Lendo a tabela silver já tratada
df_silver = spark.table("cafe_sales_2026.silver.clean_cafe_sales")  # ajuste o nome pro seu catálogo/schema

df_silver.printSchema()

In [0]:
#Aqui a ideia é: para cada coluna, contar quantas linhas têm null, "UNKNOWN", "ERROR" ou "Not Informed" (seu marcador de recuperação), e calcular o percentual.
total_rows = df_silver.count()

# Lista de "valores problemáticos" que você quer rastrear
problem_values = ["UNKNOWN", "ERROR", "Not Informed", "-"]

quality_rows = []

for col_name in df_silver.columns:
    # conta nulls
    null_count = df_silver.filter(F.col(col_name).isNull()).count()
    
    # conta valores problemáticos (só funciona em colunas string)
    if dict(df_silver.dtypes)[col_name] == "string":
        problem_count = df_silver.filter(F.col(col_name).isin(problem_values)).count()
    else:
        problem_count = 0
    
    quality_rows.append((
        col_name,
        null_count,
        problem_count,
        round((null_count + problem_count) / total_rows * 100, 2)
    ))

df_quality_by_column = spark.createDataFrame(
    quality_rows,
    ["column_name", "null_count", "problem_value_count", "pct_problematic"]
).orderBy(F.desc("pct_problematic"))

display(df_quality_by_column)

In [0]:
df_quality_by_column.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("cafe_sales_2026.gold.data_quality_by_column")